In [10]:
import gurobipy as gp
from gurobipy import GRB

# Data
store_types = ["Jewelry", "Shoe", "Department", "Book", "Clothing"]
space = {"Jewelry": 400, "Shoe": 500, "Department": 1500, "Book": 800, "Clothing": 1000}
min_stores = {"Jewelry": 1, "Shoe": 1, "Department": 1, "Book": 0, "Clothing": 2}
max_stores = {"Jewelry": 3, "Shoe": 3, "Department": 2, "Book": 2, "Clothing": 4}

# Profits per store for each quantity
profits = {
    "Jewelry": [90000, 80000, 70000, 0],
    "Shoe": [100000, 90000, 50000, 0],
    "Department": [270000, 210000, 0, 0],
    "Book": [160000, 90000, 0, 0],
    "Clothing": [0, 130000, 100000, 90000]
}

# Initialize model
model = gp.Model("Simons_Mall")

# Decision variables
x = {}
for store in store_types:
    for j in range(1, max_stores[store] + 1):
        x[store, j] = model.addVar(vtype=GRB.BINARY, name=f"x_{store}_{j}")

# Objective function: Maximize total rental income (7.5% of total profit)
model.setObjective(
    gp.quicksum(0.075 * j * profits[store][j-1] * x[store, j] for store in store_types for j in range(1, max_stores[store] + 1)),
    GRB.MAXIMIZE
)

# Constraint 1: Space limit
model.addConstr(
    gp.quicksum(space[store] * j * x[store, j] for store in store_types for j in range(1, max_stores[store] + 1)) <= 9000,
    "Space_Limit"
)

# Constraints for minimum and maximum stores
for store in store_types:
    # Only one decision for how many stores
    model.addConstr(
        gp.quicksum(x[store, j] for j in range(1, max_stores[store] + 1)) == 1,
        f"OneChoice_{store}"
    )
    # Minimum constraint
    model.addConstr(
        gp.quicksum(j * x[store, j] for j in range(1, max_stores[store] + 1)) >= min_stores[store],
        f"MinStores_{store}"
    )
    # Maximum constraint
    model.addConstr(
        gp.quicksum(j * x[store, j] for j in range(1, max_stores[store] + 1)) <= max_stores[store],
        f"MaxStores_{store}"
    )

# Solve the model
model.optimize()

# Display the results
if model.status == GRB.OPTIMAL:
    print("\nOptimal Store Allocation and Rental Income:")
    for store in store_types:
        for j in range(1, max_stores[store] + 1):
            if x[store, j].x > 0.5:
                print(f"{store}: {j} stores")
    print(f"\nTotal Annual Rental Income: ${model.objVal:.2f}")
else:
    print("No optimal solution found.")


Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-8300H CPU @ 2.30GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 14 columns and 56 nonzeros
Model fingerprint: 0x3bf418ba
Variable types: 0 continuous, 14 integer (14 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+03]
  Objective range  [7e+03, 3e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 9e+03]
Found heuristic solution: objective 89250.000000
Presolve removed 13 rows and 7 columns
Presolve time: 0.00s
Presolved: 3 rows, 7 columns, 11 nonzeros
Variable types: 0 continuous, 7 integer (7 binary)
Found heuristic solution: objective 95250.000000

Root relaxation: cutoff, 0 iterations, 0.00 seconds (0.00 work units)

Explored 1 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 8 (of 8 available processors)

Solution 

In [12]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

# Data
store_types = ["Jewelry", "Shoe", "Department", "Book", "Clothing"]
space = {"Jewelry": 400, "Shoe": 500, "Department": 1500, "Book": 800, "Clothing": 1000}
min_stores = {"Jewelry": 1, "Shoe": 1, "Department": 1, "Book": 0, "Clothing": 2}
max_stores = {"Jewelry": 3, "Shoe": 3, "Department": 2, "Book": 2, "Clothing": 4}

# Profits per store for each quantity
profits = {
    "Jewelry": [90000, 80000, 70000, 0],
    "Shoe": [100000, 90000, 50000, 0],
    "Department": [270000, 210000, 0, 0],
    "Book": [160000, 90000, 0, 0],
    "Clothing": [0, 130000, 100000, 90000]
}

# Track results for different space limits
results = []

# Vary additional square footage from 1000 to 4000 (increments of 1000)
for additional_space in range(1000, 5000, 1000):
    total_space = 9000 + additional_space

    # Initialize model
    model = gp.Model("Simons_Mall")

    # Decision variables
    x = {}
    for store in store_types:
        for j in range(1, max_stores[store] + 1):
            x[store, j] = model.addVar(vtype=GRB.BINARY, name=f"x_{store}_{j}")

    # Objective function: Maximize total rental income (7.5% of total profit)
    model.setObjective(
        gp.quicksum(0.075 * j * profits[store][j-1] * x[store, j] for store in store_types for j in range(1, max_stores[store] + 1)),
        GRB.MAXIMIZE
    )

    # Constraint 1: Space limit
    model.addConstr(
        gp.quicksum(space[store] * j * x[store, j] for store in store_types for j in range(1, max_stores[store] + 1)) <= total_space,
        "Space_Limit"
    )

    # Constraints for minimum and maximum stores
    for store in store_types:
        # Only one decision for how many stores
        model.addConstr(
            gp.quicksum(x[store, j] for j in range(1, max_stores[store] + 1)) == 1,
            f"OneChoice_{store}"
        )
        # Minimum constraint
        model.addConstr(
            gp.quicksum(j * x[store, j] for j in range(1, max_stores[store] + 1)) >= min_stores[store],
            f"MinStores_{store}"
        )
        # Maximum constraint
        model.addConstr(
            gp.quicksum(j * x[store, j] for j in range(1, max_stores[store] + 1)) <= max_stores[store],
            f"MaxStores_{store}"
        )

    # Solve the model
    model.optimize()

    # Record results if optimal
    if model.status == GRB.OPTIMAL:
        result = {"Total Space": total_space, "Total Profit": model.objVal}
        for store in store_types:
            for j in range(1, max_stores[store] + 1):
                if x[store, j].x > 0.5:
                    result[f"{store} Stores"] = j
        results.append(result)

# Convert results to DataFrame and print
df = pd.DataFrame(results)
print(df)


Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-8300H CPU @ 2.30GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 14 columns and 56 nonzeros
Model fingerprint: 0x0b27fbda
Variable types: 0 continuous, 14 integer (14 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+03]
  Objective range  [7e+03, 3e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+04]
Found heuristic solution: objective 89250.000000
Presolve removed 14 rows and 8 columns
Presolve time: 0.00s
Presolved: 2 rows, 6 columns, 8 nonzeros
Variable types: 0 continuous, 6 integer (6 binary)
Found heuristic solution: objective 96750.000000

Root relaxation: objective 9.975000e+04, 0 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbe

In [ ]:
#average profit per sqft . shop type wise and vs a whole . thus compare shops 

In [14]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

# Data
store_types = ["Jewelry", "Shoe", "Department", "Book", "Clothing"]
space = {"Jewelry": 400, "Shoe": 500, "Department": 1500, "Book": 800, "Clothing": 1000}
min_stores = {"Jewelry": 1, "Shoe": 1, "Department": 1, "Book": 0, "Clothing": 2}
max_stores = {"Jewelry": 3, "Shoe": 3, "Department": 2, "Book": 2, "Clothing": 4}

# Profits per store for each quantity
profits = {
    "Jewelry": [90000, 80000, 70000, 0],
    "Shoe": [100000, 90000, 50000, 0],
    "Department": [270000, 210000, 0, 0],
    "Book": [160000, 90000, 0, 0],
    "Clothing": [0, 130000, 100000, 90000]
}

# Track results for different space limits
results = []

# Vary additional square footage from 1000 to 4000 (increments of 1000)
for additional_space in range(1000, 6000, 1000):
    total_space = 8000 + additional_space

    # Initialize model
    model = gp.Model("Simons_Mall")

    # Decision variables
    x = {}
    for store in store_types:
        for j in range(1, max_stores[store] + 1):
            x[store, j] = model.addVar(vtype=GRB.BINARY, name=f"x_{store}_{j}")

    # Objective function: Maximize total rental income (7.5% of total profit)
    model.setObjective(
        gp.quicksum(0.075 * j * profits[store][j-1] * x[store, j] for store in store_types for j in range(1, max_stores[store] + 1)),
        GRB.MAXIMIZE
    )

    # Constraint 1: Space limit
    model.addConstr(
        gp.quicksum(space[store] * j * x[store, j] for store in store_types for j in range(1, max_stores[store] + 1)) <= total_space,
        "Space_Limit"
    )

    # Constraints for minimum and maximum stores
    for store in store_types:
        # Only one decision for how many stores
        model.addConstr(
            gp.quicksum(x[store, j] for j in range(1, max_stores[store] + 1)) == 1,
            f"OneChoice_{store}"
        )
        # Minimum constraint
        model.addConstr(
            gp.quicksum(j * x[store, j] for j in range(1, max_stores[store] + 1)) >= min_stores[store],
            f"MinStores_{store}"
        )
        # Maximum constraint
        model.addConstr(
            gp.quicksum(j * x[store, j] for j in range(1, max_stores[store] + 1)) <= max_stores[store],
            f"MaxStores_{store}"
        )

    # Solve the model
    model.optimize()

    # Record results if optimal
    if model.status == GRB.OPTIMAL:
        result = {"Total Space": total_space, "Total Profit": model.objVal}
        total_profit = 0
        total_sqft = 0
        # Calculate shop-wise profit per sqft
        for store in store_types:
            for j in range(1, max_stores[store] + 1):
                if x[store, j].x > 0.5:
                    num_stores = j
                    profit_per_store = profits[store][j-1] * num_stores
                    sqft_for_store = space[store] * num_stores
                    profit_per_sqft = profit_per_store / sqft_for_store if sqft_for_store > 0 else 0

                    result[f"{store} Stores"] = num_stores
                    result[f"{store} Profit/Sqft"] = round(profit_per_sqft, 2)

                    total_profit += profit_per_store
                    total_sqft += sqft_for_store

        # Calculate overall average profit per sqft
        avg_profit_per_sqft = total_profit / total_sqft if total_sqft > 0 else 0
        result["Overall Profit/Sqft"] = round(avg_profit_per_sqft, 2)

        results.append(result)

# Convert results to DataFrame and print
df = pd.DataFrame(results)
print(df)


Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-8300H CPU @ 2.30GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 14 columns and 56 nonzeros
Model fingerprint: 0x3bf418ba
Variable types: 0 continuous, 14 integer (14 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+03]
  Objective range  [7e+03, 3e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 9e+03]
Found heuristic solution: objective 89250.000000
Presolve removed 13 rows and 7 columns
Presolve time: 0.00s
Presolved: 3 rows, 7 columns, 11 nonzeros
Variable types: 0 continuous, 7 integer (7 binary)
Found heuristic solution: objective 95250.000000

Root relaxation: cutoff, 0 iterations, 0.00 seconds (0.00 work units)

Explored 1 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 8 (of 8 available processors)

Solution 

In [10]:
from gurobipy import Model, GRB

# Create a new model
m = Model("Cubs_Pitchers")

# Decision variables (continuous because user prefers GRB.CONTINUOUS)
players = ['RS', 'BS', 'DE', 'ST', 'TS']
costs = {'RS': 25, 'BS': 18, 'DE': 10, 'ST': 8, 'TS': 7}
victories = {'RS': 6, 'BS': 5, 'DE': 3, 'ST': 3, 'TS': 2}
righty = {'RS': 1, 'BS': 1, 'DE': 1, 'ST': 0, 'TS': 1}

# Define variables
x = {p: m.addVar(vtype=GRB.BINARY, lb=0, ub=1, name=p) for p in players}
#x = {p: m.addVar(vtype=GRB.BINARY, lb=0, ub=1, name=p) for p in players}
# Objective function: maximize victories
m.setObjective(sum(victories[p] * x[p] for p in players), GRB.MAXIMIZE)

# Constraint 1: Budget
m.addConstr(sum(costs[p] * x[p] for p in players) <= 55, "budget")

# Constraint 2: Right-handed pitchers
m.addConstr(sum(righty[p] * x[p] for p in players) <= 3, "righty_limit")

# Constraint 3: Can't sign all three RS, BS, DE
m.addConstr(x['RS'] + x['BS'] + x['DE'] <= 2, "no_three_stars")

# Optimize the model
m.optimize()

# Print results
if m.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    for p in players:
        print(f"{p}: {x[p].x:.0f}")
    print(f"Total victories: {m.objVal:.2f}")
else:
    print("No optimal solution found.")


Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-8300H CPU @ 2.30GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3 rows, 5 columns and 12 nonzeros
Model fingerprint: 0x805075a2
Variable types: 0 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [2e+00, 6e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 6e+01]
Found heuristic solution: objective 14.0000000
Presolve removed 3 rows and 5 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: 14 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.400000000000e+01, best bound 1.400000000000e+01, gap 0.0000%
Optimal solution found:
RS: 1
BS: 0
DE: 1
ST:

In [15]:
from gurobipy import Model, GRB

# Data
players = ['RS', 'BS', 'DE', 'ST', 'TS']
costs = {'RS': 25, 'BS': 18, 'DE': 10, 'ST': 8, 'TS': 7}
victories = {'RS': 6, 'BS': 5, 'DE': 3, 'ST': 3, 'TS': 2}
righty = {'RS': 1, 'BS': 1, 'DE': 1, 'ST': 0, 'TS': 1}

# Vary the budget from 40 to 68 million in steps of 4
for budget in range(40, 69, 2):
    # Create a new model for each budget
    m = Model(f"Cubs_Pitchers_Budget_{budget}")

    # Define variables (continuous as per user preference)
    x = {p: m.addVar(vtype=GRB.BINARY, name=p) for p in players}
    #x = {p: m.addVar(vtype=GRB.BINARY, lb=0, ub=1, name=p) for p in players}

    # Objective function: maximize victories
    m.setObjective(sum(victories[p] * x[p] for p in players), GRB.MAXIMIZE)

    # Constraint 1: Budget
    m.addConstr(sum(costs[p] * x[p] for p in players) <= budget, "budget")

    # Constraint 2: Right-handed pitchers
    m.addConstr(sum(righty[p] * x[p] for p in players) <= 3, "righty_limit")

    # Constraint 3: Can't sign all three RS, BS, DE
    m.addConstr(x['RS'] + x['BS'] + x['DE'] <= 2, "no_three_stars")

    # Optimize the model
    m.optimize()

    # Print results for the current budget
    print(f"\nResults for Budget = ${budget} million:")
    if m.status == GRB.OPTIMAL:
        for p in players:
            print(f"{p}: {x[p].x:.0f}")
        print(f"Total victories: {m.objVal:.2f}")
    else:
        print("No optimal solution found.")


Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-8300H CPU @ 2.30GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3 rows, 5 columns and 12 nonzeros
Model fingerprint: 0x6b42c8a8
Variable types: 0 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [2e+00, 6e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 4e+01]
Found heuristic solution: objective 9.0000000
Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 2 rows, 5 columns, 8 nonzeros
Variable types: 0 continuous, 5 integer (5 binary)

Root relaxation: objective 1.114286e+01, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0   11.1428

In [3]:
from gurobipy import Model, GRB

# Data
players = ['RS', 'BS', 'DE', 'ST', 'TS']
costs = {'RS': 25, 'BS': 18, 'DE': 10, 'ST': 8, 'TS': 7}
victories = {'RS': 6, 'BS': 5, 'DE': 3, 'ST': 3, 'TS': 2}
righty = {'RS': 1, 'BS': 1, 'DE': 1, 'ST': 0, 'TS': 1}

# Vary the budget from 40 to 68 million and right-handed pitchers from 2 to 4
for budget in range(40, 69, 4):
    for max_righty in range(2, 5):
        # Create a new model for each combination
        m = Model(f"Cubs_Pitchers_Budget_{budget}_Righty_{max_righty}")

        # Define variables (continuous as per user preference)
        x = {p: m.addVar(vtype=GRB.BINARY, name=p) for p in players}

        # Objective function: maximize victories
        m.setObjective(sum(victories[p] * x[p] for p in players), GRB.MAXIMIZE)

        # Constraint 1: Budget
        m.addConstr(sum(costs[p] * x[p] for p in players) <= budget, "budget")

        # Constraint 2: Right-handed pitchers
        m.addConstr(sum(righty[p] * x[p] for p in players) <= max_righty, "righty_limit")

        # Constraint 3: Can't sign all three RS, BS, DE
        m.addConstr(x['RS'] + x['BS'] + x['DE'] <= 2, "no_three_stars")

        # Optimize the model
        m.optimize()

        # Print results for the current budget and right-handed pitcher limit
        print(f"\nResults for Budget = ${budget} million and Righty Limit = {max_righty}:")
        if m.status == GRB.OPTIMAL:
            for p in players:
                print(f"{p}: {x[p].x:.0f}")
            print(f"Total victories: {m.objVal:.2f}")
        else:
            print("No optimal solution found.")


Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-8300H CPU @ 2.30GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3 rows, 5 columns and 12 nonzeros
Model fingerprint: 0x22086d17
Variable types: 0 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [2e+00, 6e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 4e+01]
Found heuristic solution: objective 9.0000000
Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 2 rows, 5 columns, 9 nonzeros
Variable types: 0 continuous, 5 integer (5 binary)

Root relaxation: objective 1.114286e+01, 2 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0   11.1428